<a href="https://colab.research.google.com/github/pleshro/GlassesGAN_release/blob/main/Glasses_Try_on_App_demo_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Initalization tasks

In [ ]:
#@title ### Load Necessary Packages

import os
from getpass import getpass
import urllib
import shutil
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
import gdown

### Functions to use Webcam in Google colab
from IPython.display import display, Javascript
from IPython.display import Image as display_image
from google.colab.output import eval_js
from base64 import b64decode

In [ ]:
#@title ### Helper functions

def clone_public_repo(repo_name, repo_url, base_working_dir, overwrite=False):

  new_path = f'{base_working_dir}/{repo_name}'

  #Remove old directory
  if overwrite:
    if os.path.isdir(new_path):
      print(f'Removing old directory ({new_path})')
      shutil.rmtree(new_path)

  if os.path.isdir(new_path): #If dir already exists, leave alone
    os.chdir(new_path)
    print(f'Dir already exists ({new_path})')
  else: #clone repo
    os.chdir(base_working_dir)
    cmd = f'git clone {repo_url} {new_path}'
    !{cmd}
    os.chdir(new_path)
  return

def plot_images_subplot(images, #list of images or image paths
                        plots_tall_wide=None, #tuple with number of subplots tall and wide
                        sub_titles=None,
                        figsize = (8,10), #(width, tall)
                        dpi = 500,
                        sharex = 'all',
                        sharey = 'all',
                        title='Image Subplots',
                        remove_ticks=False,
                        save_path_name = None,
                        fill_images_columnwise=False,
                        max_images_limit=100,
                        subplot_title_text_size=10,
                        use_tight_padding=False,
                        use_col_row_names=False,
                        col_names=None,
                        row_names=None,
                        col_name_font_weight=None,
                        col_name_font_size=None,
                        row_name_font_weight=None,
                        row_name_font_size=None):


    #protect mem consumption
    if len(images) > max_images_limit:
        raise Exception(f'Danger: too many images, {len(images)} > {max_images_limit}, stopping')

    #if filepaths are given, extract images
    if type(images[0]) == str:
        images = [plt.imread(fp) for fp in images]

    if len(images[0].shape) == 2:
        cmap='gray'
    else:
        cmap=None


    if not plots_tall_wide:
        plots_tall = int(np.ceil(np.sqrt(len(images))))
        plots_wide = plots_tall
    else:
        plots_tall, plots_wide = plots_tall_wide

    if not sub_titles:
        if use_col_row_names:
          sub_titles = ['' for i in range(len(images))]
        else:
          sub_titles = [f'{i}' for i in range(len(images))]

    if fill_images_columnwise:
      #Reorientent images to fill column first
      a = np.arange(len(images))
      a = a.reshape((plots_tall, plots_wide), order='f')
      new_ordering = [a.flatten()[i] for i in range(a.size)]
      images = [images[i] for i in new_ordering]
      sub_titles = [sub_titles[i] for i in new_ordering]


    fig, axs = plt.subplots(plots_tall,
                            plots_wide,
                            sharex=sharex,
                            sharey=sharey,
                            dpi=dpi,
                            figsize = figsize,
                            frameon=True)

    fig.suptitle(title)

    if plots_tall == 1:
        plot_locs = [ j for j in range(plots_wide) ]
    elif plots_wide == 1:
        plot_locs = [ i for i in range(plots_tall) ]
    else:
        plot_locs = [(i,j) for i in range(plots_tall) for j in range(plots_wide)]

    for i, im in enumerate(images):
        plt.axes(axs[plot_locs[i]])
        plt.imshow(im,cmap=cmap)
        plt.title(sub_titles[i], fontsize=subplot_title_text_size)
        if remove_ticks:
            plt.gca().axes.get_xaxis().set_ticks([])
            plt.gca().axes.get_yaxis().set_ticks([])

    if use_col_row_names:
      if col_name_font_weight is None:
        col_name_font_weight = rcParams['axes.titleweight']
      if col_name_font_size is None:
        col_name_font_size = rcParams['axes.titlesize']
      for ax, col in zip(axs[0,:], col_names):
          ax.set_title(col,
                       fontdict={'fontsize': col_name_font_size,
                                 'fontweight': col_name_font_weight},
                       loc='center',
                       pad=6.0) #offset from top of axis

      if row_name_font_weight is None:
        row_name_font_weight = rcParams['axes.titleweight']
      if row_name_font_size is None:
        row_name_font_size = rcParams['axes.titlesize']
      for ax, row in zip(axs[:,0], row_names):
          ax.set_ylabel(row,
                        rotation=90,
                        fontdict={'fontsize': col_name_font_size,
                                 'fontweight': col_name_font_weight})
                        # size='large')

    if use_tight_padding:
        plt.tight_layout(pad=0.01, h_pad=None, w_pad=None, rect=None)

    if save_path_name is None:
        plt.show()
    else:
        plt.savefig(save_path_name)

    return

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [ ]:
#@title ### Download repo, download model files, and install ninja

base_working_dir = '/content'
repo_name_dir = 'GlassesGAN_release'

### Download GlassesGAN repository
repo_name = f'{repo_name_dir}'
repo_url = f'https://github.com/pleshro/{repo_name_dir}.git'
clone_public_repo(repo_name=repo_name,
                  repo_url=repo_url,
                  base_working_dir=base_working_dir,
                  overwrite=False)



def download_gdrive_file(file_id, output_path):
    if os.path.exists(output_path):
        # Check if the file is extremely small (likely a failed HTML redirect download)
        if os.path.getsize(output_path) < 100000:
            print(f"Removing corrupted file: {output_path}")
            os.remove(output_path)
        else:
            print(f"File already exists and seems valid: {output_path}")
            return

    url = f'https://drive.google.com/uc?id={file_id}'
    print(f"Downloading {output_path} from Google Drive...")
    gdown.download(url, output_path, quiet=False)

download_gdrive_file('1rn2HJXLn1qC8c-6OARrWPw7IFbddt9eo', f'{base_working_dir}/deeplab_epoch_19.pth')
download_gdrive_file('1h7AGjY4EZDNVLJe2fLkXhrHdLUJzNRiS', f'{base_working_dir}/aveglassesdiff_celebhq_dataset_results_SG.npy')
download_gdrive_file('1FXTPEkyaSFN_IRZq04_pF-iD0qnyPgwf', f'{base_working_dir}/aveglassesdiff_celebhq_dataset_results_RG.npy')
download_gdrive_file('1XeGJ6bMulBwtRj7PvK6NSRHN16E3yCsv', f'{base_working_dir}/fitted_pca_celebhq_dataset_results.joblib')
download_gdrive_file('1lydmMcVZUV-tr6gKp8Tv-icD3uvAwCED', f'{base_working_dir}/e4e_ffhq_encode.pt')

if not os.path.exists('ninja-linux.zip'):
    !wget https://github.com/ninja-build/ninja/releases/download/v1.8.2/ninja-linux.zip
    !sudo unzip -o ninja-linux.zip -d /usr/local/bin/
    !sudo update-alternatives --install /usr/bin/ninja ninja /usr/local/bin/ninja 1 --force

### Load glasses_vton_inference class
os.chdir(f'{base_working_dir}/{repo_name_dir}')
from glasses_vton_inference import glasses_vton_inference
os.chdir(f'{base_working_dir}/{repo_name_dir}')


### Choose Tinted or Clear Glasses

In [ ]:
### Set parameters for trained model


GlassesType = 'Clear Glasses' #@param ["Clear Glasses", "Tinted Glasses"]


# train_dataset_folder_name='11_pair_private_dataset_results'
# train_dataset_folder_name='celebhq_dataset_results'

# pca_type = 'kpca'
# pca_type = 'pca'

glasses_option = 'RG' if GlassesType == 'Clear Glasses' else 'SG'
# glasses_option = 'SG'


# num_train_samples = 2
# outer_dilation_factor=5
# outer_blur_factor=25
# inner_blur_factor=5
# ideal_avg_glasses_frame_area = 0.015
# chosen_deeplab_epoch = 19 #19

# num_train_samples = 7
outer_dilation_factor=5
outer_blur_factor=25//2
inner_blur_factor=5
ideal_avg_glasses_frame_area = 0.020 #0.018 #0.022
chosen_deeplab_epoch = 19


load_loc = base_working_dir

use_full_glasses_or_frames_mask = 'frames' if glasses_option == 'RG' else 'full_glasses'

fitted_pca_fp = f'{load_loc}/fitted_pca_celebhq_dataset_results.joblib'


ave_add_glasses_diff_fp = f'{load_loc}/aveglassesdiff_celebhq_dataset_results_{glasses_option}.npy'
# DatasetGAN_input_ouput_fp = f'DatasetGAN_input_ouput_files_{num_train_samples}_glasses_sample'
resume_model_ckpt = f'{load_loc}/deeplab_epoch_{chosen_deeplab_epoch}.pth'

gvton = glasses_vton_inference(
                     fitted_pca_fp = fitted_pca_fp,
                     ave_add_glasses_diff_fp = ave_add_glasses_diff_fp,
                     base_working_dir = base_working_dir,
                     temp_save_folder = f'{base_working_dir}/tempfolder',
                     resume_model_ckpt = resume_model_ckpt,
                     deeplab_script_location = (f'{base_working_dir}/{repo_name_dir}/'
                                                'datasetGAN_release/datasetGAN'),
                     chosen_deeplab_epoch=chosen_deeplab_epoch,
                     load_loc=load_loc,
                     use_full_glasses_or_frames_mask=use_full_glasses_or_frames_mask,
                     run_tests=False,
                     outer_dilation_factor=outer_dilation_factor,
                     outer_blur_factor=outer_blur_factor,
                     inner_blur_factor=inner_blur_factor,
                     ideal_avg_glasses_frame_area=ideal_avg_glasses_frame_area,
                     auto_clean=True)




#Inference

Choose picture source for editing. "Uploaded File" uses the image stored at '/content/photo.jpg', while WebCam activiates the camera on the device to take a picture.

In [ ]:
picture_source = 'Uploaded File' #@param ["Uploaded File", "WebCam"]

In [ ]:
#@title Take Webcam Image and Initialize: Click "Run Cell" button below to take new webcam photo

if picture_source == 'WebCam':
  image_path = f'{base_working_dir}/capture.jpg'
  try:
    filename = take_photo(filename=f'{base_working_dir}/capture.jpg')
    print('Saved to {}'.format(filename))

    # Show the image which was just taken.
    display(display_image(filename))
  except Exception as err:
    # Errors will be thrown if the user does not have a webcam or if they do not
    # grant the page permission to access it or if not using google colab
    print(str(err))
else:
  image_path = f'{base_working_dir}/{repo_name_dir}/target.png'
  plt.figure()
  plt.imshow(plt.imread(image_path))


input_image, result_image, base_latent = gvton.embed_input_image(image_path,
                                                                show_plots=False)

# Ensure base_latent is float32
base_latent = base_latent.astype('float32')
if hasattr(gvton, 'ave_add_glasses_diff'):
  gvton.ave_add_glasses_diff = gvton.ave_add_glasses_diff.astype('float32')

# Monkey-patch run_generator to enforce float32 and prevent any upcasted arrays from failing the assertion
original_run_generator = gvton.e4e.run_generator
def patched_run_generator(latent_code):
  if isinstance(latent_code, np.ndarray):
    latent_code = latent_code.astype('float32')
  return original_run_generator(latent_code)

gvton.e4e.run_generator = patched_run_generator

start_latent, edit_image, _ = gvton.add_avg_glasses(input_image=input_image,
                                        base_latent=base_latent,
                                        base_bias=1,
                                        show_plots=False,
                                        auto_pick_bias=True,
                                        return_blended_image=True)

# Glasses Personalization
Select values and click "Run Cell" button on left on sliders

In [ ]:
#@title { display-mode: "form"}

%matplotlib inline

Size = 6 #@param {type:"slider", min:-5.0, max:10.0, step:0.5}
Height = -9 #@param {type:"slider", min:-10.0, max:4.0, step:0.5}
Squareness = 8 #@param {type:"slider", min:-4.0, max:20.0, step:0.5}
Round_Shrink = 0 #@param {type:"slider", min:-20.0, max:20.0, step:0.5}
Cateye = 1.5 #@param {type:"slider", min:-15.0, max:20.0, step:0.5}
Thicken = 6.5 #@param {type:"slider", min:-10.0, max:10.0, step:0.5}

Size = -Size
Height = -Height
Cateye = -Cateye

num_pcs = 6
latent = start_latent.copy()
for PC_num, PC_bias in enumerate((Size, Height, Squareness, Round_Shrink, Cateye, Thicken)):
  run_gen = True if PC_num == (num_pcs-1) else False
  img, latent = gvton.e4e.run_gen_add_pc_direction_bias(start_latent = latent,
                                                fitted_pca = gvton.fitted_pca,
                                                bias = PC_bias,
                                                PC_num = PC_num,
                                                run_gen = run_gen)

blends, _, _ = gvton.blend_in_edits(edits=[img], input_image=input_image)

# print(type(input_image))
# print(type(edit_image))
# print(type( blends[0]))

plot_images_subplot(images = [np.array(input_image), edit_image, blends[0]], #list of images or image paths
                    plots_tall_wide=(1,3), #tuple with number of subplots tall and wide
                    sub_titles=('Input', 'Normal Glasses', 'Customized Glasses'),
                    figsize = (10,4), #(width, tall)
                    dpi = 150,
                    sharex = 'all',
                    sharey = 'all',
                    title='',
                    remove_ticks=True,
                    save_path_name = None,
                    fill_images_columnwise=False,
                    max_images_limit=100,
                    subplot_title_text_size=10,
                    use_tight_padding=True,
                    use_col_row_names=False,
                    col_names=None,
                    row_names=None,
                    col_name_font_weight=None,
                    col_name_font_size=None,
                    row_name_font_weight=None,
                    row_name_font_size=None)

